# Engagement 2: Mobile-money Agent Data Cleanup

## Data-quality audit

The audit records each detected issue by dimension, column, and affected-row count before analysis. Raw data is loaded with explicit date handling and the generator is the reproducible source of the deliberately injected issues.

In [ ]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv('agent_transactions.csv', keep_default_na=True)
agents = pd.read_csv('agents.csv')
print('Raw transaction rows:', len(txns))
print('Agent rows:', len(agents))
print('Raw types:', sorted(txns['txn_type'].dropna().unique()))
print('Raw amount missing:', txns['amount'].isna().sum())
clean_df, cleaning_log = clean_agent_data(txns, agents)
audit_table = pd.DataFrame(cleaning_log, columns=['issue', 'dimension', 'column', 'rows_affected'])
audit_table

## Cleaning decisions

Duplicate transaction IDs are reduced to one record per ID. Transaction types are stripped, lowercased, mapped to exactly `Deposit` or `Withdrawal`, and invalid or missing types would be excluded. Amounts are coerced to numeric; missing or invalid values are imputed with the median of valid amounts and marked in `amount_imputed`, preserving rows while making the treatment visible. Dates accept both hyphen and slash separators and are saved as ISO dates. IQR outliers are retained and flagged because an extreme value may be a genuine transaction requiring business investigation rather than automatic deletion. Unknown agents are retained after a left merge with a missing county so the governance issue remains countable.

## Post-merge validation

In [ ]:
validation = pd.DataFrame({
    'measure': ['Raw transaction rows', 'Clean transaction rows', 'Unique txn_id values', 'Unknown-agent transactions', 'Duplicate txn_id values after cleaning'],
    'value': [len(txns), len(clean_df), clean_df['txn_id'].nunique(), clean_df['county'].isna().sum(), clean_df['txn_id'].duplicated().sum()]
})
validation

## County summary

In [ ]:
county_summary = clean_df.groupby(['county', 'txn_type'], dropna=False).agg(
    total_value=('amount', 'sum'),
    number_of_transactions=('txn_id', 'nunique'),
    average_value=('amount', 'mean')
).reset_index()
county_summary

## Weekly transaction-value pivot

In [ ]:
clean_df['week'] = pd.to_datetime(clean_df['txn_date']).dt.to_period('W').astype(str)
weekly_pivot = clean_df.pivot_table(
    index='week', columns='county', values='amount', aggfunc='sum', fill_value=0, dropna=False
)
weekly_pivot

In [ ]:
clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)
print('Exported rows:', len(clean_df))
print('txn_type values:', sorted(clean_df['txn_type'].unique()))
print('ISO dates:', clean_df['txn_date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all())

# Data Quality Report

## Uniqueness
The transaction extract contains repeated `txn_id` values. The pipeline keeps the first record for each ID, logs the number removed, and verifies that the cleaned output has no duplicate IDs.

## Consistency
Transaction-type spelling and case variants are standardised to exactly `Deposit` and `Withdrawal`. Missing or unrecognised values are identified before mapping and would be excluded rather than silently converted.

## Validity
Amounts are converted to numeric values and dates are parsed after normalising slash separators, then written in ISO `YYYY-MM-DD` form. IQR fences identify suspicious amounts; the KES 250,000 records are retained and flagged for operational investigation because deleting potentially genuine transactions would lose value.

## Completeness
Missing and non-numeric amounts are imputed with the median of valid amounts so valid transaction rows are not discarded. The `amount_imputed` flag preserves a transparent record of every replacement.

## Accuracy
The county and weekly summaries use the cleaned amount field, while outlier flags allow managers to compare reported totals with and without suspicious transactions. The pipeline is deterministic and records every treatment in its audit log.

## Referential integrity
A left join preserves every cleaned transaction and adds the agent county where the ID exists in the register. Unknown IDs remain visible with a missing county and are counted in post-merge validation rather than being silently dropped.

## Privacy
If customer phone numbers were present, they would be personal data under the Kenya Data Protection Act 2019. The business should collect them only for a defined lawful purpose, minimise access, encrypt them in transit and at rest, mask or pseudonymise them for analysis, restrict exports, and delete them when the retention purpose ends.